# Migrate an Azure Machine Learning command job to Microsoft Foundry

## Objective

Use the included `aml-foundry-migrate` CLI to:

- analyze an existing Azure Machine Learning command job without changing Azure resources;
- identify supported, adapted, unsupported, and permission-dependent capabilities;
- migrate supported code, data, model, environment, identity, compute, and output bindings to Microsoft Foundry Jobs; and
- choose copied data assets or zero-copy references to source storage.

This preview sample uses Azure Machine Learning jobs and assets, Microsoft Foundry project Jobs, Dataset and Model APIs, managed identity, and Azure RBAC.

## Time

Allow about 5 minutes to install the package, configure the notebook, and run read-only analysis. Migration time depends on asset sizes, compute availability, and the duration of the migrated command job.

## About this example

The migration tool is analyzer-first and fail-closed. It creates nothing during `analyze`, and `migrate` runs the same compatibility and permission checks before transferring assets or submitting a target job. The qualified scope is standalone, single-instance command jobs; distributed jobs, pipelines, sweeps, AutoML, environment builds, and interactive services require manual work.

All cloud operations in this notebook are disabled by default. Review the generated command and replace every placeholder before opting in.

### Data and model assets

No dataset is bundled with this sample. The analyzer discovers the code, data, model, environment, and output dependencies of the selected Azure Machine Learning command job. Upload mode copies supported assets to Foundry project storage; reference mode leaves supported data in source storage.

## Before you begin

You need:

- an Azure subscription and an existing Azure Machine Learning workspace with a completed or reusable command job;
- a Microsoft Foundry project, project storage connection, compatible Foundry compute, and a user-assigned managed identity attached to the project;
- permission to read the source job and assets and create target assets and jobs; and
- an authenticated Azure CLI session (`az login` and `az account set --subscription <subscription-id>`).

For reference mode, also create a Foundry project connection to the source storage account. RBAC inspection is read-only unless you explicitly add `--grant-reference-storage-access`.

## Installation

Install the package from this sample directory. The Azure AI Projects preview build used by Foundry Jobs is hosted on the Azure SDK public development feed.

In [ ]:
%pip install --quiet --extra-index-url https://pkgs.dev.azure.com/azure-sdk/public/_packaging/azure-sdk-for-python/pypi/simple -e .

## Parameters

Replace every placeholder. Keep both run flags `False` until you have reviewed the commands printed below.

In [ ]:
source_subscription_id = "<subscription-id>"
source_resource_group = "<aml-resource-group>"
source_workspace = "<aml-workspace>"
source_export_compute = "<aml-compute-name>"
source_job_name = "<aml-command-job-name>"

project_endpoint = "https://<account>.services.ai.azure.com"
project_name = "<foundry-project>"
storage_connection = "<project-storage-connection>"
foundry_compute_id = "<foundry-compute-resource-id>"
foundry_instance_type = "Singularity.D4_v3"
target_uai_resource_id = "<target-uai-resource-id>"

dataset_transfer_mode = "upload"  # "upload" or "reference"
source_storage_connection = ""  # required only for reference mode
analysis_policy = "migratable"
grant_reference_storage_access = False

run_analysis = False
run_migration = False

## Build and inspect the commands

The commands call the package through the active Python interpreter. `analyze` is read-only. `migrate` transfers assets and submits a Foundry job only after the selected preflight policy passes.

In [ ]:
import shlex
import subprocess
import sys

module = "foundrytrainingjob.aml_command_job_migration_cli"
common_args = [
    "--source-subscription",
    source_subscription_id,
    "--source-resource-group",
    source_resource_group,
    "--source-workspace",
    source_workspace,
    "--source-export-compute",
    source_export_compute,
    "--project-endpoint",
    project_endpoint,
    "--project-name",
    project_name,
    "--storage-connection",
    storage_connection,
    "--dataset-transfer-mode",
    dataset_transfer_mode,
    "--foundry-compute-id",
    foundry_compute_id,
    "--foundry-instance-type",
    foundry_instance_type,
    "--user-assigned-identity-id",
    target_uai_resource_id,
]
if dataset_transfer_mode == "reference":
    common_args.extend(["--source-storage-connection", source_storage_connection])

analysis_command = [
    sys.executable,
    "-m",
    module,
    "analyze",
    *common_args,
    "--source-job",
    source_job_name,
    "--analysis-policy",
    analysis_policy,
    "--report-file",
    "analysis-report.json",
]
migration_command = [
    sys.executable,
    "-m",
    module,
    "migrate",
    *common_args,
    "--source-job",
    source_job_name,
    "--work-dir",
    "migration-run",
    "--preflight-policy",
    analysis_policy,
]
if grant_reference_storage_access:
    migration_command.append("--grant-reference-storage-access")

print("Analyze:\n", shlex.join(analysis_command))
print("\nMigrate:\n", shlex.join(migration_command))

## Analyze before writing

Set `run_analysis = True` after replacing the placeholders. Review `analysis-report.json`, especially `policyPassed`, blocking capability IDs, adaptations, and runtime permission findings.

## Run the migration

Set `run_migration = True` only after the analysis is acceptable. Upload mode copies supported data to project storage. Reference mode registers source-storage URIs without copying data and requires a matching Foundry connection. Automatic Blob Data Reader assignment remains disabled unless `grant_reference_storage_access = True`.

In [ ]:
import re

placeholder_pattern = re.compile(r"<[^<>]+>")


def validate_configuration(command: list[str]) -> None:
    unresolved = sorted(
        {value for value in command if placeholder_pattern.search(value)}
    )
    if unresolved:
        raise ValueError(
            "Replace all placeholders before running a cloud command: "
            + ", ".join(unresolved)
        )


if run_analysis:
    validate_configuration(analysis_command)
    subprocess.run(analysis_command, check=True)
else:
    print("Analysis is disabled. Set run_analysis=True after configuration.")

if run_migration:
    validate_configuration(migration_command)
    subprocess.run(migration_command, check=True)
else:
    print("Migration is disabled. Set run_migration=True after reviewing analysis.")